# nSpikeTrainExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/nSpikeTrainExamples.md`


Notebook source link: [nSpikeTrainExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/nSpikeTrainExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "nSpikeTrainExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"nSpikeTrainExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"nSpikeTrainExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"nSpikeTrainExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"nSpikeTrainExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# nSpikeTrainExamples: spike-train resampling and signal representations.
from nstat.compat.matlab import nspikeTrain

spike_times = np.sort(rng.random(100))
spike_times = np.unique(np.round(spike_times * 10000.0) / 10000.0)
nst = nspikeTrain(spike_times=spike_times, t_start=0.0, t_end=1.0, name="n1")
orig_spike_count = int(nst.getSpikeTimes().size)

fig, axes = plt.subplots(4, 1, figsize=(9.0, 7.4), sharex=False)
plt.sca(axes[0])
nst.plot()
axes[0].set_title(f"{TOPIC}: original spike train")
axes[0].set_xlabel("time [s]")

nst.resample(1.0 / 0.1)
sig_100ms = nst.getSigRep(binSize_s=0.1, mode="binary")
axes[1].step(np.arange(sig_100ms.size) * 0.1, sig_100ms, where="post", color="tab:blue")
axes[1].set_title("100 ms representation")

nst.resample(1.0 / 0.01)
sig_10ms = nst.getSigRep(binSize_s=0.01, mode="binary")
axes[2].step(np.arange(sig_10ms.size) * 0.01, sig_10ms, where="post", color="tab:green")
axes[2].set_title("10 ms representation")

max_bin = float(max(nst.getMaxBinSizeBinary(), 1.0e-3))
nst.resample(1.0 / max_bin)
sig_max = nst.getSigRep(binSize_s=max_bin, mode="binary")
axes[3].step(np.arange(sig_max.size) * max_bin, sig_max, where="post", color="tab:red")
axes[3].set_title("max binary bin-size representation")
axes[3].set_xlabel("time [s]")
plt.tight_layout()
plt.show()

assert orig_spike_count > 20
assert 0.0 < max_bin <= 1.0

CHECKPOINT_METRICS = {
    "num_spikes_initial": float(orig_spike_count),
    "num_spikes_final": float(nst.getSpikeTimes().size),
    "max_bin_size": float(max_bin),
}
CHECKPOINT_LIMITS = {
    "num_spikes_initial": (20.0, 150.0),
    "num_spikes_final": (1.0, 150.0),
    "max_bin_size": (1.0e-4, 1.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
